# AI Comic Translator para Google Colab

Notebook listo para traducir páginas de cómics del inglés al español, con OCR, traducción, limpieza de texto original, renderizado y exportación en ZIP.

Flujo principal:
1. Subes imágenes o un ZIP.
2. El notebook detecta texto con OCR.
3. Traduce automáticamente.
4. Borra el texto original con inpainting.
5. Reescribe la traducción dentro de los globos.
6. Guarda y descarga un ZIP con el resultado.

In [ ]:
# 1) Instalar dependencias necesarias
import sys
import subprocess
import importlib.util

def ensure_package(package_name, module_name=None):
    module_name = module_name or package_name.replace('-', '_')
    if importlib.util.find_spec(module_name) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package_name])

packages = {
    'easyocr': 'easyocr',
    'opencv-python': 'cv2',
    'pillow': 'PIL',
    'numpy': 'numpy',
    'deep-translator': 'deep_translator',
    'matplotlib': 'matplotlib',
    'requests': 'requests',
    'beautifulsoup4': 'bs4',
    'pytesseract': 'pytesseract',
    'tqdm': 'tqdm',
}

for package_name, module_name in packages.items():
    ensure_package(package_name, module_name)

try:
    subprocess.run(['apt-get', 'update'], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    subprocess.run(['apt-get', 'install', '-y', 'tesseract-ocr'], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
except Exception as exc:
    print('Instalación opcional de Tesseract omitida:', exc)

print('Dependencias listas.')

In [ ]:
# 2) Imports y configuración general
import os
import re
import math
import json
import shutil
import zipfile
from pathlib import Path
from collections import defaultdict

import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont
from tqdm.auto import tqdm
import requests
from bs4 import BeautifulSoup
from deep_translator import GoogleTranslator
import easyocr

try:
    import pytesseract
except Exception:
    pytesseract = None

try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    files = None
    IN_COLAB = False

OUTPUT_ROOT = Path('/content/comic_translation_output') if IN_COLAB else Path('comic_translation_output')
INPUT_DIR = OUTPUT_ROOT / 'input'
OUTPUT_DIR = OUTPUT_ROOT / 'translated'
DEBUG_DIR = OUTPUT_ROOT / 'debug'

for folder in [INPUT_DIR, OUTPUT_DIR, DEBUG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

CONFIG = {
    'source_lang': 'en',
    'target_lang': 'es',
    'keep_original_subtitle': True,
    'use_gpu': False,
    'ocr_min_confidence': 0.25,
    'padding': 8,
    'inpaint_radius': 3,
    'max_font_size': 46,
    'min_font_size': 12,
    'subtitle_ratio': 0.55,
    'show_previews': True,
    'preview_pages': 1,
    'output_zip_name': 'comic_translated_pages.zip',
    'use_url_input': False,
    'input_url': '',
}

SUPPORTED_TARGET_LANGS = {
    'es': 'Spanish',
    'en': 'English',
    'fr': 'French',
    'pt': 'Portuguese',
    'it': 'Italian',
    'de': 'German',
}

IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
TRANSLATION_CACHE = {}
OCR_READER = None
FONT_PATH_CACHE = None

print('Entorno listo.')

In [ ]:
# 3) Utilidades de entrada: subir ZIP o imágenes, o descargar desde URL
def clear_folder(folder_path):
    folder_path = Path(folder_path)
    if folder_path.exists():
        for child in folder_path.iterdir():
            if child.is_file() or child.is_symlink():
                child.unlink()
            elif child.is_dir():
                shutil.rmtree(child)
    folder_path.mkdir(parents=True, exist_ok=True)

def is_image_file(path):
    return Path(path).suffix.lower() in IMAGE_EXTENSIONS

def guess_extension(content_type, fallback_url=''):
    content_type = (content_type or '').lower()
    fallback_url = fallback_url.lower()
    if 'png' in content_type or fallback_url.endswith('.png'):
        return '.png'
    if 'webp' in content_type or fallback_url.endswith('.webp'):
        return '.webp'
    if 'bmp' in content_type or fallback_url.endswith('.bmp'):
        return '.bmp'
    return '.jpg'

def extract_zip_file(zip_path, destination):
    extracted = []
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(destination)
    for file_path in Path(destination).rglob('*'):
        if file_path.is_file() and is_image_file(file_path):
            extracted.append(file_path)
    return extracted

def collect_image_files(root_folder):
    files_found = []
    for file_path in sorted(Path(root_folder).rglob('*')):
        if file_path.is_file() and is_image_file(file_path):
            files_found.append(file_path)
    unique_files = []
    seen = set()
    for file_path in files_found:
        resolved = str(file_path.resolve())
        if resolved not in seen:
            unique_files.append(file_path)
            seen.add(resolved)
    return unique_files

def download_images_from_url(url, destination):
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    headers = {'User-Agent': 'Mozilla/5.0'}
    response = requests.get(url, timeout=30, headers=headers)
    response.raise_for_status()

    content_type = response.headers.get('content-type', '').lower()
    saved_paths = []

    if 'image' in content_type or re.search(r'\.(jpg|jpeg|png|webp|bmp)(\?|$)', url.lower()):
        extension = guess_extension(content_type, url)
        output_path = destination / f'downloaded_001{extension}'
        output_path.write_bytes(response.content)
        return [output_path]

    soup = BeautifulSoup(response.text, 'html.parser')
    candidate_urls = []
    for image_tag in soup.find_all('img'):
        source = image_tag.get('data-src') or image_tag.get('src') or image_tag.get('data-lazy-src')
        if not source:
            continue
        full_url = requests.compat.urljoin(url, source)
        if full_url not in candidate_urls:
            candidate_urls.append(full_url)

    if not candidate_urls:
        for link_tag in soup.find_all('a', href=True):
            href = requests.compat.urljoin(url, link_tag['href'])
            if re.search(r'\.(jpg|jpeg|png|webp|bmp)(\?|$)', href.lower()):
                candidate_urls.append(href)

    if not candidate_urls:
        raise ValueError('No se encontraron imágenes en la URL proporcionada.')

    for index, image_url in enumerate(candidate_urls, start=1):
        try:
            image_response = requests.get(image_url, timeout=30, headers=headers)
            image_response.raise_for_status()
            extension = guess_extension(image_response.headers.get('content-type', ''), image_url)
            output_path = destination / f'url_{index:03d}{extension}'
            output_path.write_bytes(image_response.content)
            saved_paths.append(output_path)
        except Exception as exc:
            print(f'No se pudo descargar {image_url}: {exc}')

    return saved_paths

def prepare_input_images():
    clear_folder(INPUT_DIR)

    if CONFIG['use_url_input'] and CONFIG['input_url'].strip():
        print('Descargando imágenes desde URL...')
        downloaded = download_images_from_url(CONFIG['input_url'].strip(), INPUT_DIR)
        return sorted(downloaded)

    if IN_COLAB:
        if files is None:
            raise RuntimeError('No se pudo acceder a google.colab.files.')
        print('Sube imágenes JPG/PNG o un ZIP con páginas del cómic.')
        uploaded = files.upload()
        if not uploaded:
            raise ValueError('No se subió ningún archivo.')

        for filename, file_bytes in uploaded.items():
            output_path = INPUT_DIR / filename
            output_path.write_bytes(file_bytes)
            if output_path.suffix.lower() == '.zip':
                extract_zip_file(output_path, INPUT_DIR)

        return collect_image_files(INPUT_DIR)

    print('Modo local detectado. Se usarán las imágenes dentro de la carpeta input.')
    return collect_image_files(INPUT_DIR)

In [ ]:
# 4) OCR, traducción, limpieza y renderizado
def clean_text(text):
    text = '' if text is None else str(text)
    text = re.sub(r'\s+', ' ', text).strip()
    replacements = {
        ' .': '.',
        ' ,': ',',
        ' !': '!',
        ' ?': '?',
        ' ;': ';',
        ' :': ':',
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    return text.strip()


def get_reader():
    global OCR_READER
    if OCR_READER is None:
        use_gpu = False
        try:
            import torch
            use_gpu = bool(CONFIG['use_gpu']) and torch.cuda.is_available()
        except Exception:
            use_gpu = False
        OCR_READER = easyocr.Reader(['en'], gpu=use_gpu, verbose=False)
    return OCR_READER


def bbox_to_rect(bbox):
    points = np.array(bbox, dtype=np.float32)
    xs = points[:, 0]
    ys = points[:, 1]
    return int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())


def rect_to_bbox(rect):
    x1, y1, x2, y2 = rect
    return np.array([[x1, y1], [x2, y1], [x2, y2], [x1, y2]], dtype=np.int32)


def expand_rect(rect, padding, width, height):
    x1, y1, x2, y2 = rect
    return (
        max(0, x1 - padding),
        max(0, y1 - padding),
        min(width - 1, x2 + padding),
        min(height - 1, y2 + padding),
    )


def detect_with_tesseract(image_bgr):
    if pytesseract is None:
        return []

    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    data = pytesseract.image_to_data(image_rgb, lang='eng', output_type=pytesseract.Output.DICT)
    grouped = defaultdict(list)

    for index, raw_text in enumerate(data['text']):
        text = clean_text(raw_text)
        try:
            confidence = float(data['conf'][index])
        except Exception:
            confidence = -1.0

        if not text or confidence < 40:
            continue

        key = (data['block_num'][index], data['par_num'][index], data['line_num'][index])
        x = int(data['left'][index])
        y = int(data['top'][index])
        w = int(data['width'][index])
        h = int(data['height'][index])
        grouped[key].append((x, y, w, h, text, confidence))

    items = []
    for group in grouped.values():
        xs = [entry[0] for entry in group]
        ys = [entry[1] for entry in group]
        xe = [entry[0] + entry[2] for entry in group]
        ye = [entry[1] + entry[3] for entry in group]
        text = clean_text(' '.join(entry[4] for entry in group))
        if not text:
            continue
        items.append({
            'bbox': rect_to_bbox((min(xs), min(ys), max(xe), max(ye))),
            'text': text,
            'confidence': float(np.mean([entry[5] for entry in group])),
            'engine': 'tesseract',
        })

    items.sort(key=lambda item: (bbox_to_rect(item['bbox'])[1], bbox_to_rect(item['bbox'])[0]))
    return items


def detect_text_items(image_bgr):
    reader = get_reader()
    results = reader.readtext(image_bgr, detail=1, paragraph=True, min_size=10, text_threshold=0.4, low_text=0.3)
    items = []
    for bbox, text, confidence in results:
        text = clean_text(text)
        if not text or float(confidence) < float(CONFIG['ocr_min_confidence']):
            continue
        items.append({
            'bbox': np.array(bbox, dtype=np.int32),
            'text': text,
            'confidence': float(confidence),
            'engine': 'easyocr',
        })
    if not items:
        items = detect_with_tesseract(image_bgr)
    items.sort(key=lambda item: (bbox_to_rect(item['bbox'])[1], bbox_to_rect(item['bbox'])[0]))
    return items


def translate_text(text, target_lang):
    text = clean_text(text)
    if not text:
        return ''
    cache_key = (text, target_lang)
    if cache_key in TRANSLATION_CACHE:
        return TRANSLATION_CACHE[cache_key]
    try:
        translated = GoogleTranslator(source='auto', target=target_lang).translate(text)
    except Exception as exc:
        print(f'No se pudo traducir "{text}": {exc}')
        translated = text
    translated = clean_text(translated) or text
    TRANSLATION_CACHE[cache_key] = translated
    return translated


def find_font_path():
    global FONT_PATH_CACHE
    if FONT_PATH_CACHE is not None:
        return FONT_PATH_CACHE
    candidates = []
    try:
        from matplotlib import font_manager
        for font_path in font_manager.findSystemFonts(fontpaths=None, fontext='ttf'):
            lower = font_path.lower()
            score = 0
            if 'comic' in lower:
                score += 10
            if 'rounded' in lower:
                score += 5
            if 'bold' in lower:
                score += 3
            if 'dejavu' in lower or 'liberation' in lower or 'noto' in lower:
                score += 1
            candidates.append((score, font_path))
    except Exception:
        candidates = []
    candidates.sort(reverse=True, key=lambda item: item[0])
    FONT_PATH_CACHE = candidates[0][1] if candidates else None
    return FONT_PATH_CACHE


def measure_text(draw, text, font):
    sample = text if text else 'Ag'
    bbox = draw.textbbox((0, 0), sample, font=font)
    return bbox[2] - bbox[0], bbox[3] - bbox[1]


def split_long_word(draw, word, font, max_width):
    if measure_text(draw, word, font)[0] <= max_width:
        return [word]
    parts = []
    current = ''
    for char in word:
        candidate = current + char
        if not current or measure_text(draw, candidate, font)[0] <= max_width:
            current = candidate
        else:
            parts.append(current)
            current = char
    if current:
        parts.append(current)
    return parts


def wrap_text_to_width(draw, text, font, max_width):
    lines = []
    for paragraph in str(text).split('\n'):
        words = paragraph.split()
        if not words:
            lines.append('')
            continue
        current_line = ''
        for word in words:
            for segment in split_long_word(draw, word, font, max_width):
                candidate = segment if not current_line else current_line + ' ' + segment
                if not current_line or measure_text(draw, candidate, font)[0] <= max_width:
                    current_line = candidate
                else:
                    lines.append(current_line)
                    current_line = segment
        if current_line:
            lines.append(current_line)
    return lines or ['']


def measure_block(draw, lines, font, line_spacing=4):
    total_width = 0
    total_height = 0
    for index, line in enumerate(lines):
        line_width, line_height = measure_text(draw, line, font)
        total_width = max(total_width, line_width)
        total_height += line_height
        if index < len(lines) - 1:
            total_height += line_spacing
    return total_width, total_height


def choose_font_layout(draw, translated_text, original_text, box_width, box_height, font_path):
    translated_text = clean_text(translated_text)
    original_text = clean_text(original_text)
    subtitle_text = ''
    if CONFIG['keep_original_subtitle'] and original_text and original_text != translated_text:
        subtitle_text = original_text

    padded_width = max(20, box_width - 6)
    padded_height = max(20, box_height - 6)

    for main_size in range(CONFIG['max_font_size'], CONFIG['min_font_size'] - 1, -1):
        main_font = ImageFont.truetype(font_path, main_size) if font_path else ImageFont.load_default()
        main_lines = wrap_text_to_width(draw, translated_text, main_font, padded_width)
        main_width, main_height = measure_block(draw, main_lines, main_font, line_spacing=max(2, main_size // 7))
        total_height = main_height
        subtitle_lines = []
        subtitle_font = None

        if subtitle_text:
            subtitle_size = max(CONFIG['min_font_size'], int(main_size * CONFIG['subtitle_ratio']))
            subtitle_font = ImageFont.truetype(font_path, subtitle_size) if font_path else ImageFont.load_default()
            subtitle_lines = wrap_text_to_width(draw, subtitle_text, subtitle_font, padded_width)
            subtitle_width, subtitle_height = measure_block(draw, subtitle_lines, subtitle_font, line_spacing=max(1, subtitle_size // 8))
            total_height += max(2, main_size // 8) + subtitle_height
            main_width = max(main_width, subtitle_width)

        if main_width <= padded_width and total_height <= padded_height:
            return {
                'main_font': main_font,
                'main_lines': main_lines,
                'subtitle_font': subtitle_font,
                'subtitle_lines': subtitle_lines,
                'subtitle_text': subtitle_text,
            }

    main_font = ImageFont.truetype(font_path, CONFIG['min_font_size']) if font_path else ImageFont.load_default()
    subtitle_font = ImageFont.truetype(font_path, max(CONFIG['min_font_size'] - 2, 8)) if (font_path and subtitle_text) else (ImageFont.load_default() if subtitle_text else None)
    main_lines = wrap_text_to_width(draw, translated_text, main_font, padded_width)
    subtitle_lines = wrap_text_to_width(draw, subtitle_text, subtitle_font, padded_width) if subtitle_text else []
    return {
        'main_font': main_font,
        'main_lines': main_lines,
        'subtitle_font': subtitle_font,
        'subtitle_lines': subtitle_lines,
        'subtitle_text': subtitle_text,
    }


def draw_centered_multiline(draw, box, lines, font, fill=(0, 0, 0), line_spacing=4):
    x1, y1, x2, y2 = box
    box_width = x2 - x1
    box_height = y2 - y1
    total_width, total_height = measure_block(draw, lines, font, line_spacing=line_spacing)
    start_x = x1 + max(0, (box_width - total_width) / 2)
    start_y = y1 + max(0, (box_height - total_height) / 2)
    current_y = start_y
    for index, line in enumerate(lines):
        line_width, line_height = measure_text(draw, line, font)
        current_x = x1 + max(0, (box_width - line_width) / 2)
        stroke_width = max(1, getattr(font, 'size', CONFIG['min_font_size']) // 18)
        draw.text((current_x, current_y), line, font=font, fill=fill, stroke_width=stroke_width, stroke_fill='white')
        current_y += line_height
        if index < len(lines) - 1:
            current_y += line_spacing


def inpaint_text_regions(image_bgr, items):
    mask = np.zeros(image_bgr.shape[:2], dtype=np.uint8)
    height, width = mask.shape
    for item in items:
        x1, y1, x2, y2 = expand_rect(bbox_to_rect(item['bbox']), CONFIG['padding'], width, height)
        cv2.rectangle(mask, (x1, y1), (x2, y2), 255, -1)
    return cv2.inpaint(image_bgr, mask, CONFIG['inpaint_radius'], cv2.INPAINT_TELEA), mask


def annotate_ocr_preview(image_bgr, items):
    preview = cv2.cvtColor(image_bgr.copy(), cv2.COLOR_BGR2RGB)
    for index, item in enumerate(items, start=1):
        bbox = np.array(item['bbox'], dtype=np.int32)
        cv2.polylines(preview, [bbox], isClosed=True, color=(255, 0, 0), thickness=2)
        x1, y1, x2, y2 = bbox_to_rect(bbox)
        label = f"{index}: {item['text'][:28]}"
        cv2.putText(preview, label, (x1, max(14, y1 - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 0, 0), 1, cv2.LINE_AA)
    return preview


def render_translations(image_bgr, items, font_path=None):
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    pil_image = Image.fromarray(image_rgb)
    draw = ImageDraw.Draw(pil_image)
    for item in items:
        x1, y1, x2, y2 = bbox_to_rect(item['bbox'])
        layout = choose_font_layout(draw, item.get('translated', item['text']), item['text'], x2 - x1, y2 - y1, font_path)
        draw_centered_multiline(draw, (x1, y1, x2, y2), layout['main_lines'], layout['main_font'], fill=(0, 0, 0), line_spacing=max(2, getattr(layout['main_font'], 'size', CONFIG['min_font_size']) // 7))
        if layout['subtitle_text'] and layout['subtitle_lines'] and layout['subtitle_font'] is not None:
            main_block_width, main_block_height = measure_block(draw, layout['main_lines'], layout['main_font'], line_spacing=max(2, getattr(layout['main_font'], 'size', CONFIG['min_font_size']) // 7))
            subtitle_gap = max(2, getattr(layout['main_font'], 'size', CONFIG['min_font_size']) // 8)
            subtitle_total_width, subtitle_total_height = measure_block(draw, layout['subtitle_lines'], layout['subtitle_font'], line_spacing=max(1, getattr(layout['subtitle_font'], 'size', CONFIG['min_font_size']) // 8))
            subtitle_start_y = y1 + max(0, ((y2 - y1) - (main_block_height + subtitle_gap + subtitle_total_height)) / 2) + main_block_height + subtitle_gap
            current_y = subtitle_start_y
            for index, line in enumerate(layout['subtitle_lines']):
                line_width, line_height = measure_text(draw, line, layout['subtitle_font'])
                current_x = x1 + max(0, ((x2 - x1) - line_width) / 2)
                draw.text((current_x, current_y), line, font=layout['subtitle_font'], fill=(80, 80, 80), stroke_width=0, stroke_fill='white')
                current_y += line_height
                if index < len(layout['subtitle_lines']) - 1:
                    current_y += max(1, getattr(layout['subtitle_font'], 'size', CONFIG['min_font_size']) // 8)
    return cv2.cvtColor(np.array(pil_image), cv2.COLOR_RGB2BGR)
